# 02 — NextTrack Demo

End-to-end demo: given recent track IDs, `recommend()` returns ranked next-track
recommendations with tag-based `why` explanations. Stateless — no user identity,
the recent tracks are the only input.

Run order: train first (`uv run train`) so `prototypes/artifacts/` exists, then
Run All here.

In [1]:
import pickle
from pathlib import Path

from nextrack import recommend
from nextrack.train import IDMAP_PATH

maps = pickle.load(open(IDMAP_PATH, 'rb'))
names = maps['track_names']
tags = maps['tags']
tid_to_idx = maps['track_id_to_index']
print(f'catalogue: {len(tid_to_idx):,} tracks  |  tagged: {len(tags):,}')

catalogue: 35,861 tracks  |  tagged: 24,668


E:\Education\1-University of london\01-March-September 26 session\01-CM3070 Computer Science Final Project\project\code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Seed: Pink Floyd — *The Dark Side of the Moon*

A tight, canonical cluster: 5 tracks from one album. Its Last.fm tags are
genre-dominated (progressive rock, psychedelic, classic rock), so the `why`
explanations read cleanly. We hardcode the track IDs (verified present in the
trained catalogue) rather than auto-selecting.

In [2]:
# Pink Floyd - Dark Side of the Moon (2011 Remastered), verified in catalogue.
seeds = [
    '41454059',  # Time
    '7741243',   # Breathe (In The Air)
    '26878430',  # Money
    '43041154',  # Us And Them
    '7631440',   # Brain Damage
]
for tid in seeds:
    a, t = names[tid]
    print(f'  {tid}: {a} - {t}  | tags: {tags[tid][:5]}')

  41454059: Pink Floyd - Time - 2011 Remastered Version  | tags: ['Progressive rock', 'Psychedelic Rock', 'classic rock', 'rock', 'ambient']
  7741243: Pink Floyd - Breathe (In The Air) - 2011 Remastered Version  | tags: ['Psychedelic Rock', 'classic rock', 'rock', 'Progressive rock', 'psychedelic']
  26878430: Pink Floyd - Money - 2011 Remastered Version  | tags: ['rock', 'Progressive rock', 'classic rock', 'psychedelic', 'art rock']
  43041154: Pink Floyd - Us And Them - 2011 Remastered Version  | tags: ['classic rock', 'rock', 'Progressive rock', 'psychedelic', 'art rock']
  7631440: Pink Floyd - Brain Damage - 2011 Remastered Version  | tags: ['rock', 'classic rock', 'Progressive rock', 'psychedelic', 'Psychedelic Rock']


## Get recommendations

In [3]:
recs = recommend(seeds, k=10)
for i, r in enumerate(recs, 1):
    print(f"{i:2}. {r['artist']} - {r['title']}  (score {r['score']:.3f})")
    if r['why']:
        print(f"      {r['why']}  [{', '.join(r['shared_tags'])}]")

 1. Pink Floyd - On The Run - 2011 Remastered Version  (score 0.927)
      Because you listened to genre: progressive rock, genre: psychedelic rock and genre: space rock  [genre: progressive rock, genre: psychedelic rock, genre: space rock]
 2. Pink Floyd - The Great Gig In The Sky - 2011 Remastered Version  (score 0.902)
      Because you listened to rock, psychedelic and classic rock  [rock, psychedelic, classic rock]
 3. Pink Floyd - Speak To Me - 2011 Remastered Version  (score 0.901)
      Because you listened to Progressive rock, genre: progressive rock and genre: psychedelic rock  [Progressive rock, genre: progressive rock, genre: psychedelic rock]
 4. Pink Floyd - Hey You - 2011 Remastered Version  (score 0.803)
      Because you listened to rock, Progressive rock and classic rock  [rock, Progressive rock, classic rock]
 5. Pink Floyd - Comfortably Numb - 2011 Remastered Version  (score 0.800)
      Because you listened to rock, Progressive rock and classic rock  [rock, Progres